In [ ]:
# initialize notebook
PROJECT_ID  = "qwiklabs-gcp-02-e659e41ff1eb"
LOCATION    = "US"
DATASET_ID  = "response_times_ds"
TABLE_ID    = "emergency_calls_raw"
MODEL_ID    = "response_time_model"

import os, uuid
from google.cloud import bigquery
from google.cloud import bigquery_storage

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
bqstorageclient = bigquery_storage.BigQueryReadClient()
print("BigQuery client ready.")

BigQuery client ready.


In [ ]:
# create dataset
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
client.delete_dataset(dataset_ref, delete_contents=True, not_found_ok=True)
client.create_dataset(dataset_ref)
print(f"Dataset `{DATASET_ID}` is ready.")

Dataset `response_times_ds` is ready.


In [ ]:
# Load the CSV from GCS
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
)
uri = "gs://labs.roitraining.com/data-to-ai-workshop/emergency_calls_response_times.csv"

load_job = client.load_table_from_uri(
    uri, f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}", job_config=job_config
)
load_job.result()
print(f"Loaded {load_job.output_rows:,} rows into `{TABLE_ID}`.")

Loaded 50,000 rows into `emergency_calls_raw`.


In [ ]:
#   Quick schema / preview
preview = client.query(f"""
SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
LIMIT 5
""").to_dataframe()
preview

,call_id,call_timestamp,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available,response_time
0,35957,2023-01-01 00:05:53+00:00,Fire,Highland,Rainy,Sunday,0,High,21.45,3,23.41
1,20832,2023-01-01 00:20:47+00:00,Fire,Oakmont,Rainy,Sunday,0,High,22.29,6,20.11
2,27949,2023-01-01 00:33:27+00:00,Fire,Riverside,Windy,Sunday,0,High,17.19,14,19.75
3,20199,2023-01-01 00:48:29+00:00,Fire,Riverside,Windy,Sunday,0,High,17.39,14,20.76
4,46938,2023-01-01 00:50:44+00:00,Rescue,Brookfield,Sunny,Sunday,0,High,22.50,14,22.37


In [44]:
# Create a BigQuery ML model
create_model_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_ID}`
OPTIONS(
  model_type='linear_reg',
  input_label_cols=['response_time'],
  data_split_method='AUTO_SPLIT'  -- 80/20 train-eval
) AS
SELECT
  * EXCEPT(call_id, call_timestamp, location)
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
--where 'response_time' IS NOT NULL;
"""

job = client.query(create_model_sql)
job.result()
print(f"Model `{MODEL_ID}` built ✓")

Model `response_time_model` built ✓


In [45]:
# Evaluate the model
eval_df = client.query(f"""
SELECT *
FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_ID}`)
""").to_dataframe()
eval_df

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,1.742121,4.770662,0.014883,1.474349,0.829923,0.829955


In [47]:
 # Predict on synthetic rows
  predict_df = client.query(f"""
WITH synthetic AS (
  (SELECT 7  AS time_of_day,
         'Saturday'   AS day_of_week,
         3.2 AS distance_to_station,
         2   AS units_available,
         'Fire'   AS call_type,
         'High' AS traffic_level,
         'Rainy' AS weather_condition)
  UNION ALL
  (SELECT 14, 'Tuesday', 1.1, 1, 'Police', 'Low', 'Sunny')
)
SELECT *
FROM ML.PREDICT(MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_ID}`, TABLE synthetic)
""").to_dataframe()
predict_df

,predicted_response_time,time_of_day,day_of_week,distance_to_station,units_available,call_type,traffic_level,weather_condition
0,15.288845,7,Saturday,3.2,2,Fire,High,Rainy
1,7.547045,14,Tuesday,1.1,1,Police,Low,Sunny
